In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("/data/raw")
OUT = Path("/data/interim")
OUT.mkdir(parents=True, exist_ok=True)

# ── Country mapping (R4V names → ISO3) ───────────────────────────────────────
COUNTRY_MAP = {
    "Dominican Republic":  "DOM",
    "Guyana":              "GUY",
    "Trinidad & Tobago":   "TTO",
    "Trinidad and Tobago": "TTO",
    "Costa Rica":          "CRI",
    "Panama":              "PAN",
    "Colombia":            "COL",
}

TIER1 = {
    "BLZ","CRI","CUB","DOM","SLV","GTM","GUY",
    "HTI","HND","JAM","NIC","PAN","PRI","SUR","TTO","BHS","BRB"
}


# ── Helper ────────────────────────────────────────────────────────────────────
def expand_monthly(iso3, year, annual_total, source):
    """Expand a single annual total into 12 equal monthly rows."""
    return [
        {
            "iso3": iso3,
            "year": year,
            "month": m,
            "venezuelan_displaced": round(annual_total / 12, 0),
            "displacement_source": source,
        }
        for m in pd.date_range(f"{year}-01-01", f"{year}-12-01", freq="MS")
    ]


# ─────────────────────────────────────────────────────────────────────────────
# YEARS 2019 + 2020 + 2021
# Source: population-projection-rmrp-2021.xlsx
#   - Dec2020_proj column → used for both 2019 (backfill) and 2020
#   - Total column        → used for 2021
# This is the earliest consistent population projection available from R4V.
# No RMRP 2019 or 2020 population projection file exists on HDX.
# ─────────────────────────────────────────────────────────────────────────────
def load_2019_2020_2021():
    path = RAW / "population-projection-rmrp-2021.xlsx"
    df = pd.read_excel(path, sheet_name="Pop Projections - RMRP 2021", header=0)

    # Row 0 contains real column headers
    df.columns = df.iloc[0]
    df = df.iloc[1:].reset_index(drop=True)
    df.columns = [
        "Country", "Population_group", "Stock_Sep2020",
        "Proj_2020", "Difference", "Dec2020_proj",
        "Girls", "Boys", "Women", "Men", "Total"
    ]

    df["Country"]      = df["Country"].str.strip()
    df                 = df[df["Country"].isin(COUNTRY_MAP)].copy()
    df["iso3"]         = df["Country"].map(COUNTRY_MAP)
    df["Dec2020_proj"] = pd.to_numeric(df["Dec2020_proj"], errors="coerce").fillna(0)
    df["Total"]        = pd.to_numeric(df["Total"],        errors="coerce").fillna(0)

    annual = df.groupby("iso3")[["Dec2020_proj", "Total"]].sum().reset_index()

    rows = []
    for _, r in annual.iterrows():
        rows += expand_monthly(r.iso3, 2019, r.Dec2020_proj, "r4v_backfill_dec2020_proj")
        rows += expand_monthly(r.iso3, 2020, r.Dec2020_proj, "r4v_dec2020_proj")
        rows += expand_monthly(r.iso3, 2021, r.Total,        "r4v")

    result = pd.DataFrame(rows)
    print(f"  2019-2021: {len(result)} rows, {result['iso3'].nunique()} countries")
    return result


# ─────────────────────────────────────────────────────────────────────────────
# YEAR 2022
# Source: population-projections-rmrp-2022.xlsx, sheet "RMRP2022 Projection"
# Admin1-level rows summed to country total, then expanded monthly.
# ─────────────────────────────────────────────────────────────────────────────
def load_2022():
    path = RAW / "population-projections-rmrp-2022.xlsx"
    df = pd.read_excel(path, sheet_name="RMRP2022 Projection")

    df["Country"] = df["Country"].str.strip()
    df            = df[df["Country"].isin(COUNTRY_MAP)].copy()
    df["iso3"]    = df["Country"].map(COUNTRY_MAP)
    df["Total"]   = pd.to_numeric(df["Total"], errors="coerce").fillna(0)

    annual = df.groupby("iso3")["Total"].sum().reset_index()

    rows = []
    for _, r in annual.iterrows():
        rows += expand_monthly(r.iso3, 2022, r.Total, "r4v")

    result = pd.DataFrame(rows)
    print(f"  2022: {len(result)} rows, {result['iso3'].nunique()} countries")
    return result


# ─────────────────────────────────────────────────────────────────────────────
# YEAR 2023
# Source: master-regional_population-pins-targets_2023-2024.xlsx, "Projections"
# Country-level rows identified by 2-character ISOCode.
# ─────────────────────────────────────────────────────────────────────────────
def load_2023():
    path = RAW / "master-regional_population-pins-targets_2023-2024.xlsx"
    df = pd.read_excel(path, sheet_name="Projections")

    df                 = df[df["ISOCode"].str.len() == 2].copy()
    df["Country"]      = df["Country"].str.strip()
    df                 = df[df["Country"].isin(COUNTRY_MAP)].copy()
    df["iso3"]         = df["Country"].map(COUNTRY_MAP)
    df["annual_total"] = pd.to_numeric(df["Total 2023"], errors="coerce").fillna(0)

    rows = []
    for _, r in df.iterrows():
        rows += expand_monthly(r.iso3, 2023, r.annual_total, "r4v")

    result = pd.DataFrame(rows)
    print(f"  2023: {len(result)} rows, {result['iso3'].nunique()} countries")
    return result


# ─────────────────────────────────────────────────────────────────────────────
# YEAR 2024
# Primary source: master_population-projections-pin-and-taget-2024.xlsx
# Fallback for CRI: "Total 2024" column from the 2023/2024 combined file.
#   CRI is absent from the standalone 2024 file entirely.
# ─────────────────────────────────────────────────────────────────────────────
def load_2024():
    path24 = RAW / "master_population-projections-pin-and-taget-2024.xlsx"
    df24   = pd.read_excel(path24, sheet_name="Population Projection ")

    df24              = df24[df24["ISOCode"].str.len() == 2].copy()
    df24["Country"]   = df24["Country"].str.strip()
    df24              = df24[df24["Country"].isin(COUNTRY_MAP)].copy()
    df24["iso3"]      = df24["Country"].map(COUNTRY_MAP)
    df24["annual_total"] = pd.to_numeric(
        df24["Total Population Projection 2024"], errors="coerce"
    ).fillna(0)

    rows    = []
    covered = set()
    for _, r in df24.iterrows():
        rows += expand_monthly(r.iso3, 2024, r.annual_total, "r4v")
        covered.add(r.iso3)

    # Fallback: any country in our map missing from the standalone file
    missing = set(COUNTRY_MAP.values()) - covered
    if missing:
        path23 = RAW / "master-regional_population-pins-targets_2023-2024.xlsx"
        df23   = pd.read_excel(path23, sheet_name="Projections")
        df23   = df23[df23["ISOCode"].str.len() == 2].copy()
        df23["Country"] = df23["Country"].str.strip()
        df23   = df23[df23["Country"].isin(COUNTRY_MAP)].copy()
        df23["iso3"] = df23["Country"].map(COUNTRY_MAP)

        for iso3 in missing:
            row = df23[df23["iso3"] == iso3]
            if not row.empty:
                total = float(pd.to_numeric(row.iloc[0]["Total 2024"], errors="coerce"))
                rows += expand_monthly(iso3, 2024, total, "r4v_from_2023-24_combined")
                print(f"    {iso3} 2024: fallback from combined file ({total:,.0f})")
            else:
                print(f"    WARNING: {iso3} 2024 missing from all sources")

    result = pd.DataFrame(rows)
    print(f"  2024: {len(result)} rows, {result['iso3'].nunique()} countries")
    return result


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
def main():
    print("Loading R4V population projection files...")
    df = pd.concat(
        [load_2019_2020_2021(), load_2022(), load_2023(), load_2024()],
        ignore_index=True
    )

    df["month"] = pd.to_datetime(df["month"])
    df["venezuelan_displaced"] = (
        pd.to_numeric(df["venezuelan_displaced"], errors="coerce")
        .round(0)
        .astype("Int64")
    )
    df = df.sort_values(["iso3", "month"]).reset_index(drop=True)

    # ── QA summary ────────────────────────────────────────────────────────────
    print("\n=== QA SUMMARY ===")
    print(f"Total rows   : {len(df)}")
    print(f"Countries    : {sorted(df['iso3'].unique())}")
    print(f"Date range   : {df['month'].min().date()} → {df['month'].max().date()}")
    print(f"Nulls        : {df['venezuelan_displaced'].isna().sum()}")
    print("\nAnnual totals by country (sum of monthly values):")
    pivot = (
        df.groupby(["iso3", "year"])["venezuelan_displaced"]
        .sum()
        .unstack("year")
        .fillna(0)
        .astype(int)
    )
    print(pivot.to_string())

    # ── Coverage check ────────────────────────────────────────────────────────
    print("\n=== TIER 1 COVERAGE ===")
    covered     = sorted(set(df["iso3"].unique()) & TIER1)
    not_covered = sorted(TIER1 - set(df["iso3"].unique()))
    print(f"  WITH R4V data    : {covered}")
    print(f"  WITHOUT R4V data : {not_covered}")
    print(f"  → uncovered countries fall back to unhcr_country_month.csv")
    print(f"  COL as pressure input: {'COL' in df['iso3'].unique()}")

    # ── Write outputs ─────────────────────────────────────────────────────────
    out_cols = ["iso3", "year", "month", "venezuelan_displaced", "displacement_source"]
    df[out_cols].to_csv(OUT / "r4v_country_month.csv", index=False)
    print(f"\n  Wrote r4v_country_month.csv  ({len(df)} rows)")

    r4v_year = (
        df.groupby(["iso3", "year"])["venezuelan_displaced"]
        .sum()
        .reset_index()
        .rename(columns={"venezuelan_displaced": "annual_venezuelan_displaced"})
    )
    r4v_year.to_csv(OUT / "r4v_country_year.csv", index=False)
    print(f"  Wrote r4v_country_year.csv   ({len(r4v_year)} rows)")


if __name__ == "__main__":
    main()

Loading R4V population projection files...
  2019-2021: 216 rows, 6 countries
  2022: 72 rows, 6 countries
  2023: 72 rows, 6 countries
    CRI 2024: fallback from combined file (247,158)
  2024: 72 rows, 6 countries

=== QA SUMMARY ===
Total rows   : 432
Countries    : ['COL', 'CRI', 'DOM', 'GUY', 'PAN', 'TTO']
Date range   : 2019-01-01 → 2024-12-01
Nulls        : 0

Annual totals by country (sum of monthly values):
year     2019     2020     2021      2022     2023     2024
iso3                                                       
COL   4489260  4489260  5095536  10802988  7438332  7630140
CRI     30000    30000    32700     38712   246120   247164
DOM    114048   114048   120948    121476   135180   139824
GUY     23304    23304    30300     28920    28452    29976
PAN    121836   121836   127644    131844   488484   513600
TTO     24168    24168    30768     34116    43728    44760

=== TIER 1 COVERAGE ===
  WITH R4V data    : ['CRI', 'DOM', 'GUY', 'PAN', 'TTO']
  WITHOUT R4V dat